# InputLayer Query Patterns

This notebook demonstrates three query patterns using InputLayer with data from the Reasoning Notebook demo:

1. **Semantic Retrieval** — similarity search over vector embeddings
2. **Structured Retrieval** — multi-hop reasoning with IQL rules
3. **Hybrid Queries** — combining vector similarity with rule-based reasoning

The data comes from notes (text) and images (multimodal extraction), showing how InputLayer unifies structured and unstructured queries.

In [ ]:
import sys
sys.path.insert(0, '../backend')

from inputlayer import InputLayer, Relation, Vector, HnswIndex
from inputlayer.integrations.langchain.params import iql_literal
from config import INPUTLAYER_URL, INPUTLAYER_USER, INPUTLAYER_PASSWORD, KG_NAME

il = InputLayer(INPUTLAYER_URL, username=INPUTLAYER_USER, password=INPUTLAYER_PASSWORD)
await il.connect()
kg = il.knowledge_graph(KG_NAME)
print(f'Connected to {KG_NAME}')

In [ ]:
# Quick look at what's in the KG
notes = await kg.execute('?note(Id, Title, Content, Ca, Ua)')
entities = await kg.execute('?entity(Id, Name, Kind, Desc, Source)')
relationships = await kg.execute('?relationship(Id, Subject, Predicate, Object, Source)')

print(f'Notes: {len(notes.rows)}')
print(f'Entities: {len(entities.rows)}')
print(f'Relationships: {len(relationships.rows)}')
print()
for row in notes.rows:
    print(f'  📝 {row[1]}')
print()
kinds = {}
for row in entities.rows:
    k = row[2]
    kinds[k] = kinds.get(k, 0) + 1
print('Entity kinds:', dict(sorted(kinds.items(), key=lambda x: -x[1])))

---
## 1. Semantic Retrieval (Similarity Search)

InputLayer supports HNSW vector indexes for approximate nearest-neighbor search. We embed entity descriptions and find similar entities by meaning, not exact string matching.

This is how you'd answer: *"Find entities related to navigation"* or *"What's similar to a temple?"*

In [ ]:
import hashlib

EMBED_DIM = 32

def embed(text: str) -> list[float]:
    """Simple character n-gram embedding (demo only)."""
    text = text.lower().strip()
    vec = [0.0] * EMBED_DIM
    for i in range(len(text) - 2):
        trigram = text[i:i+3]
        h = int(hashlib.md5(trigram.encode()).hexdigest(), 16)
        vec[h % EMBED_DIM] += 1.0
    norm = sum(v * v for v in vec) ** 0.5
    return [v / norm for v in vec] if norm > 0 else [1.0 / EMBED_DIM**0.5] * EMBED_DIM

# Define embedding relation and index
class EntityEmbed(Relation):
    __relation_name__ = 'entity_embed'
    id: str
    name: str
    kind: str
    embedding: Vector

await kg.define(EntityEmbed)

# Clear and re-populate embeddings from current entities
try:
    await kg.execute('-entity_embed(I, N, K, E) <- entity_embed(I, N, K, E)')
except: pass

for row in entities.rows:
    eid, name, kind, desc = row[0], row[1], row[2], row[3]
    vec = embed(f'{name} {kind} {desc}')
    await kg.insert(EntityEmbed(id=eid, name=name, kind=kind, embedding=vec))

# Create HNSW index
try:
    await kg.execute('.index drop entity_embed_idx')
except: pass

await kg.create_index(HnswIndex(
    name='entity_embed_idx',
    relation=EntityEmbed,
    column='embedding',
    metric='cosine',
))
print(f'Indexed {len(entities.rows)} entity embeddings')

In [ ]:
# Semantic search: "Find entities related to religious architecture"
query_vec = embed('religious architecture temple sacred building')
results = await kg.vector_search(EntityEmbed, query_vec, k=5, metric='cosine')

print('Query: "religious architecture"\n')
for row in results.rows:
    data = {k.lower(): v for k, v in zip(results.columns, row)}
    dist = data.get('dist', '?')
    print(f'  [{data["kind"]}] {data["name"]}  (distance: {dist})')

In [ ]:
# Semantic search: "Find entities related to navigation and sea"
query_vec = embed('navigation sea lighthouse ocean sailing ship')
results = await kg.vector_search(EntityEmbed, query_vec, k=5, metric='cosine')

print('Query: "navigation and sea"\n')
for row in results.rows:
    data = {k.lower(): v for k, v in zip(results.columns, row)}
    dist = data.get('dist', '?')
    print(f'  [{data["kind"]}] {data["name"]}  (distance: {dist})')

In [ ]:
# Semantic search: "Find people"
query_vec = embed('person human character people')
results = await kg.vector_search(EntityEmbed, query_vec, k=5, metric='cosine')

print('Query: "people"\n')
for row in results.rows:
    data = {k.lower(): v for k, v in zip(results.columns, row)}
    dist = data.get('dist', '?')
    print(f'  [{data["kind"]}] {data["name"]}  (distance: {dist})')

---
## 2. Structured Retrieval (Multi-hop Reasoning)

InputLayer's IQL rules enable multi-hop reasoning that would require multiple round-trips in a traditional database. Define a rule once, and the engine incrementally maintains the derived facts.

This is how you'd answer: *"What is connected to X through Y?"* or *"Find all colleagues."*

In [ ]:
# Basic: all entities and their relationships
print('=== Direct relationships ===')
r = await kg.execute('?relationship(Id, Subject, Predicate, Object, Source)')
for row in r.rows[:15]:
    print(f'  {row[1]} --{row[2]}--> {row[3]}')
if len(r.rows) > 15:
    print(f'  ... and {len(r.rows) - 15} more')

In [ ]:
# Define multi-hop rules
rules = [
    # Bidirectional connection
    '+connected(A, B) <- relationship(_, A, _, B, _)',
    '+connected(A, B) <- relationship(_, B, _, A, _)',
    # Two-hop reachability (A connected to B connected to C)
    '+two_hop(A, C, Via) <- connected(A, Via), connected(Via, C), A != C, A != Via, Via != C',
    # Entities co-occurring in the same note
    '+same_note(A, B, Source) <- entity(_, A, _, _, Source), entity(_, B, _, _, Source), A != B',
]

for rule in rules:
    try:
        await kg.execute(rule)
    except: pass

print('Rules deployed.')

In [ ]:
# Multi-hop: "What can I reach in 2 hops from lighthouse?"
print('=== Two-hop connections from "lighthouse" ===')
r = await kg.execute('?two_hop("lighthouse", Target, Via)')
seen = set()
for row in r.rows:
    target, via = row[0], row[1]
    if target not in seen:
        seen.add(target)
        print(f'  lighthouse -> {via} -> {target}')

In [ ]:
# Co-occurrence: "What entities appear together in the same note?"
print('=== Entities from the same note ===')
r = await kg.execute('?same_note(A, B, Source)')
pairs = set()
for row in r.rows:
    pair = tuple(sorted([row[0], row[1]]))
    if pair not in pairs:
        pairs.add(pair)
        print(f'  {pair[0]} & {pair[1]}')
    if len(pairs) >= 15:
        print(f'  ... and more')
        break

In [ ]:
# Provenance: WHY is something connected?
print('=== Provenance: why is lighthouse connected to keeper? ===')
r = await kg._execute('.why ?connected("lighthouse", "keeper")')
trees = getattr(r, 'proof_trees', []) or []
if trees:
    tree = trees[0]
    nodes = tree.get('nodes', {}) if isinstance(tree, dict) else tree.nodes
    for nid, node in (nodes.items() if isinstance(nodes, dict) else {}):
        if isinstance(node, dict):
            conc = node.get('conclusion', {})
            print(f'  [{node.get("kind")}] {conc.get("pred")}({conc.get("args", [])})')
        else:
            print(f'  [{node.kind}] {node.conclusion.pred}({node.conclusion.args})')
else:
    print('  (no proof tree — entities may not be connected)')

---
## 3. Hybrid Queries (Vector + Rules)

The real power: combine semantic similarity with structured reasoning. Find entities similar to a concept, then traverse their relationships to discover connected knowledge.

This is how you'd answer: *"Find things related to 'water' and show what they're connected to."*

In [ ]:
# Step 1: Semantic search for a concept
query = 'water ocean sea coast'
query_vec = embed(query)
similar = await kg.vector_search(EntityEmbed, query_vec, k=3, metric='cosine')

print(f'Step 1: Semantic search for "{query}"')
seed_entities = []
for row in similar.rows:
    data = {k.lower(): v for k, v in zip(similar.columns, row)}
    name = data['name']
    seed_entities.append(name)
    print(f'  Found: {name} [{data["kind"]}]')

print(f'\nStep 2: Structured traversal from seed entities')
for entity in seed_entities:
    r = await kg.execute(f'?relationship(_, "{entity}", Pred, Object, _)')
    outgoing = [(row[1], row[2]) for row in (r.rows or [])]
    r2 = await kg.execute(f'?relationship(_, Subject, Pred, "{entity}", _)')
    incoming = [(row[0], row[1]) for row in (r2.rows or [])]
    
    if outgoing or incoming:
        print(f'\n  {entity}:')
        for pred, obj in outgoing:
            print(f'    -> {pred} -> {obj}')
        for subj, pred in incoming:
            print(f'    <- {subj} <- {pred}')

In [ ]:
# Hybrid: semantic search + two-hop reasoning
query = 'ancient culture religion'
query_vec = embed(query)
similar = await kg.vector_search(EntityEmbed, query_vec, k=3, metric='cosine')

print(f'Hybrid query: "{query}"\n')
print('Semantic matches + their 2-hop connections:\n')

for row in similar.rows:
    data = {k.lower(): v for k, v in zip(similar.columns, row)}
    name = data['name']
    print(f'  [{data["kind"]}] {name} (dist: {data.get("dist", "?")})')
    
    # Two-hop from this entity
    hops = await kg.execute(f'?two_hop("{name}", Target, Via)')
    seen = set()
    for hop_row in (hops.rows or [])[:5]:
        target = hop_row[0]
        via = hop_row[1]
        if target not in seen:
            seen.add(target)
            print(f'    2-hop: {name} -> {via} -> {target}')
    print()

---
## Multimodal Queries

The extraction pipeline unifies text and image data into the same knowledge graph. Entities from a photo of a Thai temple and entities from a written travel journal **live in the same schema** and can be queried together.

There is no separate "image query" — the ontology normalizes everything.

In [ ]:
# Show entities by source type (text vs image)
print('=== Entities by source ===')
for row in entities.rows:
    source = row[4]
    source_type = 'image' if source.startswith('img:') else 'text'
    note_id = source.replace('img:', '')
    
    # Look up note title
    nr = await kg.execute(f'?note("{note_id}", Title, C, Ca, Ua)')
    title = nr.rows[0][0] if nr.rows else note_id[:8]
    print(f'  [{source_type}] [{row[2]}] {row[1]} <- "{title}"')

In [ ]:
# Cross-modal query: find connections between image entities and text entities
print('=== Cross-modal connections ===')
print('(entities from images connected to entities from text)\n')

img_entities = {row[1] for row in entities.rows if row[4].startswith('img:')}
text_entities = {row[1] for row in entities.rows if not row[4].startswith('img:')}

# Find relationships that bridge image and text entities
for row in relationships.rows:
    subj, pred, obj = row[1], row[2], row[3]
    if (subj in img_entities and obj in text_entities) or \
       (subj in text_entities and obj in img_entities):
        s_type = 'img' if subj in img_entities else 'txt'
        o_type = 'img' if obj in img_entities else 'txt'
        print(f'  [{s_type}] {subj} --{pred}--> [{o_type}] {obj}')

# Also check same_note co-occurrence
shared = img_entities & text_entities
if shared:
    print(f'\nEntities appearing in both image and text extraction:')
    for name in shared:
        print(f'  {name}')

In [ ]:
# Semantic search across modalities
# "Find everything related to dragons" — should find image-extracted dragon sculptures
# AND any text mentions of dragons
query = 'dragon sculpture mythical creature'
query_vec = embed(query)
results = await kg.vector_search(EntityEmbed, query_vec, k=5, metric='cosine')

print(f'Cross-modal semantic search: "{query}"\n')
for row in results.rows:
    data = {k.lower(): v for k, v in zip(results.columns, row)}
    # Look up source type
    er = await kg.execute(f'?entity("{data["id"]}", N, K, D, Source)')
    if er.rows:
        source = er.rows[0][4]
        source_type = 'image' if source.startswith('img:') else 'text'
    else:
        source_type = '?'
    print(f'  [{source_type}] [{data["kind"]}] {data["name"]}  (dist: {data.get("dist", "?")})')

In [ ]:
await il.close()
print('Disconnected.')